# 대응안 승인 기록

Operations agent가 Teams로 보낸 제안을 담당자가 승인(**Proceed** → **Confirm**)하면 이 Notebook이 실행됩니다.
Eventhouse의 `RiskEventStatus`에 `approved` 상태를 추가하고, Lakehouse에 승인 내역을 남깁니다.
발주나 이송 지시는 하지 않습니다.

## 1. 매개 변수

Operations agent가 실행할 때 `event_id`와 `option_id`를 넘깁니다. 아래 값은 직접 실행할 때 쓰는 기본값입니다.

In [ ]:
event_id = "EVT-20261001-001"
option_id = "OPT-2"

## 2. 위험 이벤트 확인

`gold.fact_risk_event`에서 승인할 위험 이벤트를 읽습니다.

**예상 결과:** 1행. `EVT-20261001-001`, `SO-10322`, `BNK-L3-2`, 추천안 `OPT-2`

In [ ]:
from pyspark.sql import functions as F

event = spark.table("gold.fact_risk_event").where(F.col("event_id") == event_id)
display(event.select("event_id", "sales_order_id", "bunker_id", "required_topup_kg", "recommended_option_id", "status"))
events = event.take(2)
if len(events) != 1:
    raise ValueError(f"{event_id} 위험 이벤트가 정확히 1행이어야 합니다. 조회 행 수: {len(events)}")
ev = events[0]
if option_id != ev.recommended_option_id:
    raise ValueError(f"option_id={option_id!r}가 현재 추천안 {ev.recommended_option_id!r}와 다릅니다.")
options = (spark.table("gold.fact_response_option")
           .where((F.col("option_id") == option_id) & (F.col("scenario_id") == ev.scenario_id)).take(2))
if len(options) != 1 or not options[0].meets_all:
    raise ValueError(f"{option_id}는 현재 시나리오의 기준 충족 대응안이어야 합니다.")
if ev.status not in ("open", "approved"):
    raise ValueError(f"{event_id}의 상태 {ev.status!r}에서는 승인을 기록할 수 없습니다.")

## 3. Eventhouse에 승인 상태 추가

같은 작업 영역의 Eventhouse `eh_chipbalance`를 찾아 `RiskEventStatus`에 `approved` 행을 추가합니다.
Operations agent는 `event_status`가 `open`이 될 때만 알리므로, 이 행으로는 알림을 다시 보내지 않습니다.

In [ ]:
import pandas as pd
import requests
import sempy.fabric as fabric

EVENTHOUSE = "eh_chipbalance"
KQL_DATABASE = "eh_chipbalance"

workspace_id = fabric.get_workspace_id()
eventhouses = fabric.FabricRestClient().get(f"v1/workspaces/{workspace_id}/eventhouses").json()["value"]
matches = [e for e in eventhouses if e["displayName"] == EVENTHOUSE]
if len(matches) != 1:
    raise ValueError(f"현재 작업 영역에 {EVENTHOUSE} Eventhouse가 정확히 1개 있어야 합니다.")
kusto_uri = matches[0]["properties"]["queryServiceUri"]
headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("kusto")}


def kql_text(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kusto(endpoint, command):
    response = requests.post(f"{kusto_uri}/v1/rest/{endpoint}", headers=headers,
                             json={"db": KQL_DATABASE, "csl": command}, timeout=60)
    response.raise_for_status()
    result = response.json()
    if result.get("Exceptions"):
        raise RuntimeError(f"Kusto 요청 실패: {result['Exceptions']}")
    for result_table in result.get("Tables", []):
        for row in result_table.get("Rows", []):
            values = [row] if isinstance(row, dict) else row
            for value in values:
                if isinstance(value, dict) and value.get("OneApiErrors"):
                    raise RuntimeError(f"Kusto 부분 실패: {value['OneApiErrors']}")
    table = result["Tables"][0]
    return pd.DataFrame(table["Rows"], columns=[c["ColumnName"] for c in table["Columns"]])


kusto("mgmt", f""".set-or-append RiskEventStatus <|
print event_id = {kql_text(ev.event_id)}, status_time = now(), event_status = 'approved',
      bunker_id = {kql_text(ev.bunker_id)}, sales_order_id = {kql_text(ev.sales_order_id)},
      required_topup_kg = tolong({int(ev.required_topup_kg)}),
      first_below_safety_date = datetime({ev.first_below_safety_date:%Y-%m-%d}),
      recommended_option_id = {kql_text(ev.recommended_option_id)},
      recommended_action = {kql_text(ev.recommended_action)}""")
print(f"{event_id}: RiskEventStatus에 approved 추가")

## 4. Lakehouse에 승인 기록 남기기

`dbo.chip_decision_log`에 승인 내역을 남기고, `gold.fact_risk_event`의 `status`를 `approved`로 바꿉니다.

In [ ]:
from datetime import datetime, timezone

from delta.tables import DeltaTable

decided_at = datetime.now(timezone.utc).replace(tzinfo=None)

spark.sql("""
CREATE TABLE IF NOT EXISTS dbo.chip_decision_log (
    event_id STRING,
    option_id STRING,
    decision STRING,
    decided_at TIMESTAMP
)
""")
(spark.createDataFrame([(event_id, option_id, "approved", decided_at)],
                       "event_id STRING, option_id STRING, decision STRING, decided_at TIMESTAMP")
      .write.mode("append").saveAsTable("dbo.chip_decision_log"))

DeltaTable.forName(spark, "gold.fact_risk_event").update(
    condition=F.col("event_id") == event_id,
    set={"status": F.lit("approved")},
)
print(f"{event_id}: {option_id} 승인 기록 완료 ({decided_at:%Y-%m-%d %H:%M:%S} UTC)")

## 5. 결과 확인

**예상 결과:** `RiskEventStatus`에 같은 `event_id`의 `open` 행과 `approved` 행이 시간 순서로 있습니다.
`dbo.chip_decision_log`에 승인한 행이 있고, 위험 이벤트의 `status`가 `approved`입니다.

In [ ]:
display(kusto("query", f"RiskEventStatus | where event_id == {kql_text(event_id)} "
                       "| project status_time, event_id, event_status, recommended_option_id | order by status_time asc"))
display(spark.sql("SELECT * FROM dbo.chip_decision_log ORDER BY decided_at DESC"))
display(spark.sql("SELECT event_id, sales_order_id, bunker_id, recommended_option_id, status FROM gold.fact_risk_event"))